# CeNN Optimized Memory V2 — measured competition with Transformer attention

**Run on a GPU: Runtime → Change runtime type → GPU, then Run all.** Start with `smoke` to check your environment; `balanced` is the default research screen. No result is pre-filled and no superiority is assumed.

The earlier layer-18 run was close in perplexity but substantially slower than native attention. This experiment tests two new candidates and two essential controls:

| Mixer | Purpose |
|---|---|
| `cenn_partition` | Exact sinks and recent blocks, compressed older history, one shared normalization |
| `cenn_linear` | Pure normalized recurrent memory with parallel block prefix sums |
| `sink_window` | Untrained exact sink/local attention control |
| `transformer_readout` | Full attention given the same readout calibration and training stages |

All are also compared with the untouched pretrained Transformer. Only attention mixers at selected layers change. This notebook complements the repository's newer Memory Fusion / Delta notebooks; it runs its own controlled experiment from the original pretrained model.

## Why these changes could help

For query $q_t$, let $E_t$ contain sink and recent tokens, and $G_t$ contain older non-sink tokens. These sets are disjoint. The hybrid computes

$$y_t=\frac{\sum_{i\in E_t}e^{q_t^T k_i/\sqrt d}v_i+g_t\phi(q_t)^T S_t}{\sum_{i\in E_t}e^{q_t^T k_i/\sqrt d}+g_t\phi(q_t)^T z_t},\quad S_t=\sum_{i\in G_t}\phi(k_i)v_i^T,\quad z_t=\sum_{i\in G_t}\phi(k_i).$$

Positive learned features and $g_t>0$ keep a well-defined normalized mixture. Parallel block sums remove the sequential delta update from prefill. A training-only ridge solve calibrates each head:

$$R=(Y^TY+\lambda I)^{-1}(Y^TT+\lambda I).$$

Inference folds $R$ into the frozen output projection, $O_{new}=O\operatorname{blockdiag}(R)^T$, eliminating the separate readout multiply. A stable solve is used instead of explicitly taking an inverse. These changes are hypotheses about quality and practical speed, not a proof that the candidate dominates softmax.

Research basis: [LoLCATs](https://arxiv.org/abs/2410.10254), [Sliding-window beats linear attention, August 2026](https://arxiv.org/abs/2608.28444), and conditioning motivation from [Preconditioned DeltaNet](https://arxiv.org/abs/2604.21100). This is an independent adaptation, not a full reproduction of those models. Full derivations and limitations are in [OPTIMIZED_MEMORY.md](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/OPTIMIZED_MEMORY.md).

## 1. Install into a fresh checkout

The notebook pins Transformers to its tested interface and records the exact source/model/data revisions. It uses Colab's installed PyTorch. If another notebook already imported a different Transformers version, restart the session first.

In [ ]:
import os, sys, subprocess, tempfile, json, shutil
from pathlib import Path
from datetime import datetime, timezone

REPO_REF = "main"  # Can be a tag or immutable commit for a repeat experiment.
REPO = Path(tempfile.mkdtemp(prefix="cenn-v2-")) / "TinyCeNN-LM"
subprocess.run(["git", "clone", "--quiet", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "transformers==4.57.6", "datasets>=3,<5",
                "pandas", "matplotlib", "pytest", "nbformat"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO), "--no-deps"], check=True)
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO / "src")])
import torch
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Choose the test budget and output location

`smoke` checks the pipeline and uses too few documents for a quality claim. `balanced` evaluates 12 single-layer candidates and their joint composition. `extended` adds feature widths, more training and longer contexts; it is substantially more expensive. Runtime depends on your GPU and network; no time-to-completion is guaranteed.

The default saves directly to Drive, including intermediate checkpoints and progress. Each execution gets a new directory. Re-running starts a fresh experiment; automatic training resume is not implemented.

In [ ]:
PROFILE = "balanced" # @param ["smoke", "balanced", "extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}
COMPILE_KERNELS = False # @param {type:"boolean"}
COMPUTE_DTYPE = "auto" # @param ["auto", "float32", "float16", "bfloat16"]
SEED = 2027 # @param {type:"integer"}
PROFILES = {
    "smoke": dict(layers="18", context=64, test_contexts="64,128", feature_dims="32",
                  block_size=16, train_documents=8, validation_documents=4, test_documents=4,
                  steps=10, lm_steps=2, eval_every=5),
    "balanced": dict(layers="0,18,29", context=256, test_contexts="256,512,1024", feature_dims="64",
                     block_size=32, train_documents=96, validation_documents=16, test_documents=32,
                     steps=300, lm_steps=40, eval_every=50),
    "extended": dict(layers="0,18,29", context=512, test_contexts="512,1024,2048", feature_dims="64,128",
                     block_size=32, train_documents=256, validation_documents=32, test_documents=64,
                     steps=1000, lm_steps=100, eval_every=100),
}
if not torch.cuda.is_available() and PROFILE != "smoke":
    raise RuntimeError("Select a GPU runtime, or choose smoke for a CPU pipeline check.")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/TinyCeNN/optimized-memory-v2")
else:
    OUTPUT_BASE = Path("/content/cenn-results") if Path("/content").exists() else Path.cwd() / "cenn-results"
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = OUTPUT_BASE / run_id  # Runner requires a new empty directory.
LOG = OUTPUT_BASE / (run_id + ".log")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
config = dict(PROFILES[PROFILE], sink_tokens=4, seed=SEED, compute_dtype=COMPUTE_DTYPE)
print(json.dumps(config, indent=2))
print("Results:", OUT)

## 3. Optional: exclude all documents used in previous runs

V2 always excludes V1 validation/test hash buckets. Its new holdout may overlap **V1 training documents** unless you supply previous manifests. Upload only `manifest.json` files (not checkpoint files). The runner excludes all train/validation/test hashes listed in them. It always starts from the original pretrained weights. These controls do not establish that the base model never saw this web data during pretraining.

In [ ]:
UPLOAD_PRIOR_MANIFESTS = False # @param {type:"boolean"}
PRIOR_MANIFESTS = []  # Or put existing Drive manifest paths here.
if UPLOAD_PRIOR_MANIFESTS:
    from google.colab import files
    uploaded = files.upload()
    manifest_dir = Path(tempfile.mkdtemp(prefix="prior-manifests-"))
    for index, (name, data) in enumerate(uploaded.items()):
        prior = json.loads(data)
        groups = prior.get("document_hashes")
        if not isinstance(groups, dict) or not groups:
            raise ValueError(f"{name} has no document_hashes mapping")
        path = manifest_dir / f"manifest-{index}.json"
        path.write_bytes(data)
        PRIOR_MANIFESTS.append(str(path))
print("Prior manifests:", len(PRIOR_MANIFESTS))

## 4. Preflight correctness checks

Checks include a dense independent oracle, gradients, causal streaming equivalence, bounded state storage, the ridge solve, exact readout folding, and a complete offline two-layer Llama experiment. Passing these tests establishes implementation correctness for the tested cases; it says nothing about pretrained-model quality.

In [ ]:
preflight_env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", CUDA_VISIBLE_DEVICES="")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_optimized_memory.py",
                "tests/test_optimized_memory_benchmark.py"], cwd=REPO, env=preflight_env, check=True)

## 5. Train, select on validation, then evaluate the held-out test

Only the replacement core is trained. Learned candidates and the full-attention readout control get the same transfer and language-loss update counts; their parameter counts differ. The sink control is untrained. Selection is locked before test evaluation. Layer replacements are then composed together without additional joint training.

Optional compilation is a separately labeled prefill measurement, with numerical equivalence checked and eager fallback recorded. The default decision gates use eager timings. Logs stream below and remain available if the process fails.

In [ ]:
command = [sys.executable, "-u", str(REPO / "scripts/benchmark_cenn_optimized_memory.py"),
           "--output-dir", str(OUT)]
for key, value in config.items():
    command.extend(["--" + key.replace("_", "-"), str(value)])
for path in PRIOR_MANIFESTS:
    command.extend(["--exclude-manifest", str(path)])
if COMPILE_KERNELS:
    command.append("--compile-kernels")
print(" ".join(command))
try:
    with LOG.open("w") as log:
        with subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise RuntimeError(f"Benchmark exited with code {result}; inspect {LOG}")
except BaseException as error:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "failure_report.json").write_text(json.dumps({"error": str(error), "log": str(LOG)}, indent=2))
    raise
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT / "console.log")
        archive = shutil.make_archive(str(OUT) + "-results", "zip", root_dir=OUT)
        print("Saved result archive:", archive)

## 6. Read the decision table and comparisons

- **Quality win:** upper paired 95% confidence bounds for NLL difference are below zero against **both** original and adapted Transformer controls, with at least eight test documents.
- **Quality-preserving efficiency win:** both upper NLL bounds are at most 0.02 nats/token (about 2.02% perplexity), native-SDPA prefill and decode speedups exceed 1, and state ratio is below 1.
- Ratios below 1 favor the candidate for perplexity/state; speed ratios above 1 favor the candidate. The perplexity plots are zoomed points with intervals, not bars hiding small regressions.

These are exploratory, document-level intervals from one training seed. Selection uses validation, but many comparisons still require independent confirmation. Kernel timing includes feature-map work and excludes Q/K/V/O projections for both sides. State is actual FP32 candidate storage versus native-dtype Transformer KV storage. This is not whole-model generation throughput. Joint rows separately report complete forward latency.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

report = json.loads((OUT / "optimized_memory_report.json").read_text())
if report.get("status") != "completed":
    raise RuntimeError("The run is incomplete; inspect progress.json and console.log.")
results = pd.DataFrame(report["rows"])
single = results.loc[results.scope.eq("single")].copy()
selected = single.loc[single.selected_on_validation.eq(True)]
columns = ["layer", "variant", "context", "test_perplexity", "ppl_ratio", "ppl_ratio_vs_adapted",
           "delta_nll_ci_high", "adapted_delta_nll_ci_high", "prefill_speedup", "decode_speedup",
           "state_ratio", "beats_both_quality", "quality_preserving_efficiency_win"]
print("Validation-selected bounded candidates (all test contexts):")
display(selected[columns].round(5))
selected[columns].to_csv(OUT / "selected_decisions.csv", index=False)
joint = results.loc[results.scope.eq("joint")]
if not joint.empty:
    print("Joint composition: all chosen layers replaced together")
    display(joint[["candidate", "context", "composition", "ppl_ratio", "ppl_ratio_vs_adapted",
                   "full_model_forward_speedup", "beats_both_quality"]].round(5))

for layer, part in single.groupby("layer"):
    part = part.sort_values(["context", "variant", "feature_dim"]).reset_index(drop=True)
    labels = part.apply(lambda r: f"{r.variant} · F{int(r.feature_dim)} · T{int(r.context)}", axis=1)
    y = np.arange(len(part))
    fig, axes = plt.subplots(1, 4, figsize=(21, max(4, len(part) * .36)), sharey=True)
    for ax, ratio, low, high, title in [
        (axes[0], "ppl_ratio", "delta_nll_ci_low", "delta_nll_ci_high", "PPL / original Transformer"),
        (axes[1], "ppl_ratio_vs_adapted", "adapted_delta_nll_ci_low", "adapted_delta_nll_ci_high", "PPL / adapted Transformer")]:
        lo, hi = np.exp(part[low].astype(float)), np.exp(part[high].astype(float))
        values = part[ratio].astype(float)
        ax.hlines(y, lo, hi, color="#237b9f")
        ax.scatter(values, y, s=24, color="#237b9f", zorder=3)
        margin = report["args"]["nll_margin"]
        ax.axvspan(np.exp(-margin), np.exp(margin), color="#ddd", alpha=.35)
        ax.axvline(1, color="#333", linestyle="--")
        ax.set_title(title + "\nLower is better; 95% paired intervals")
        left, right = min(float(lo.min()), np.exp(-margin)), max(float(hi.max()), np.exp(margin))
        pad = max(.003, .08 * (right - left))
        ax.set_xlim(left - pad, right + pad)
    axes[2].scatter(part.prefill_speedup, y - .13, label="Prefill / native", marker="o")
    axes[2].scatter(part.decode_speedup, y + .13, label="Decode / native", marker="s")
    axes[2].scatter(part.transformer_fp32_prefill_ms / part.prefill_ms, y, label="Prefill / FP32", marker="x")
    axes[2].axvline(1, color="#333", linestyle="--")
    axes[2].set_xscale("log")
    axes[2].set_title("Transformer time / candidate time\nHigher is better; logarithmic axis")
    axes[2].legend(fontsize=8)
    axes[3].scatter(part.state_ratio, y, color="#008577")
    axes[3].axvline(1, color="#333", linestyle="--")
    axes[3].set_xlim(left=0)
    axes[3].set_title("Candidate state / native KV bytes\nLower is better")
    axes[0].set_yticks(y, labels)
    axes[0].invert_yaxis()
    for ax in axes:
        ax.grid(axis="x", alpha=.18)
    fig.suptitle(f"Optimized memory V2 · attention layer {int(layer)}")
    fig.tight_layout()
    for suffix in ("png", "svg"):
        fig.savefig(OUT / f"comparison-layer-{int(layer)}.{suffix}", bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)
print("Completed experiment; consult the decision flags before claiming an improvement.")

## 7. Export results

Share the ZIP for review. It contains configuration/revisions, document hashes and token blocks, per-document NLL, candidate checkpoints, validation selection, progress, logs, decision tables and plots. Compare quality, actual measured speed, and memory separately. Repeat a promising configuration with additional seeds and untouched data before scaling replacement across the model.

In [ ]:
archive = shutil.make_archive(str(OUT) + "-results", "zip", root_dir=OUT)
print("Archive:", archive)
DOWNLOAD_ZIP = True # @param {type:"boolean"}
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)